In [ ]:
def delete_from_intervals(intervals, target_index):
    if not intervals:
        return []

    new_intervals = []
    running_count = 0

    for i, interval in enumerate(intervals):
        start, end = interval[0], interval[1]
        count = end - start + 1
        prev_count = running_count
        running_count += count

        # If target_index is beyond this interval, keep it unchanged
        if target_index >= running_count:
            new_intervals.append((start, end))
        # target_index is within this interval
        else:
            # Position within current interval (0-based)
            pos_in_interval = target_index - prev_count
            element_to_remove = start + pos_in_interval

            # Add left part if it exists
            if element_to_remove > start:
                new_intervals.append((start, element_to_remove - 1))

            # Add right part if it exists
            if element_to_remove < end:
                new_intervals.append((element_to_remove + 1, end))

            # Add all remaining intervals unchanged
            new_intervals.extend(intervals[i + 1:])
            break

    return new_intervals


def test_delete_from_intervals():
    # Example 1: Delete from middle (index 2 = third element = 6)
    intervals = [(4, 7), (10, 11), (13, 15)]
    result = delete_from_intervals(intervals, 2)
    print(f"Example 1: {result}")
    assert result == [(4, 5), (7, 7), (10, 11), (13, 15)]

    # Example 2: Delete from start (index 0 = first element = 4)
    intervals = [(4, 7), (10, 11), (13, 15)]
    result = delete_from_intervals(intervals, 0)
    print(f"Example 2: {result}")
    assert result == [(5, 7), (10, 11), (13, 15)]

    # Example 3: Delete from end (index 3 = fourth element = 7)
    intervals = [(4, 7), (10, 11), (13, 15)]
    result = delete_from_intervals(intervals, 3)
    print(f"Example 3: {result}")
    assert result == [(4, 6), (10, 11), (13, 15)]

    # Example 4: Delete single-element interval (index 4 = fifth element = 10)
    intervals = [(4, 7), (10, 10), (13, 15)]
    result = delete_from_intervals(intervals, 4)
    print(f"Example 4: {result}")
    assert result == [(4, 7), (13, 15)]

    print("All tests passed!")

test_delete_from_intervals()



def test_delete_from_intervals():
    # Example 1: Delete from middle
    intervals = [(4, 7), (10, 11), (13, 15)]
    result = delete_from_intervals(intervals, 2)
    print(result)
    assert result == [(4, 5), (7, 7), (10, 11), (13, 15)]

    # Example 2: Delete from start
    intervals = [(4, 7), (10, 11), (13, 15)]
    result = delete_from_intervals(intervals, 0)
    assert result == [(5, 7), (10, 11), (13, 15)]

    # Example 3: Delete from end
    intervals = [(4, 7), (10, 11), (13, 15)]
    result = delete_from_intervals(intervals, 3)
    assert result == [(4, 6), (10, 11), (13, 15)]

    # Example 4: Delete single-element interval
    intervals = [(4, 7), (10, 10), (13, 15)]
    result = delete_from_intervals(intervals, 4)
    assert result == [(4, 7), (13, 15)]

    print("All tests passed!")

test_delete_from_intervals()

Example 1: [(4, 5), (7, 7), (10, 11), (13, 15)]
Example 2: [(5, 7), (10, 11), (13, 15)]
Example 3: [(4, 6), (10, 11), (13, 15)]
Example 4: [(4, 7), (13, 15)]
All tests passed!
[(4, 5), (7, 7), (10, 11), (13, 15)]
All tests passed!


In [ ]:
## Segment Tree Implementation:
class IntervalNode:
    def __init__(self, start, end, covered_count):
        self.start = start
        self.end = end
        self.covered_count = covered_count  # Total elements in subtree
        self.left = None
        self.right = None

class IntervalTree:
    def __init__(self, intervals):
        """Build balanced tree from intervals."""
        if not intervals:
            self.root = None
        else:
            self.root = self._build(intervals, 0, len(intervals) - 1)

    def _build(self, intervals, left, right):
        """Recursively build tree."""
        if left > right:
            return None

        mid = (left + right) // 2
        start, end = intervals[mid]
        count = end - start + 1

        node = IntervalNode(start, end, count)
        node.left = self._build(intervals, left, mid - 1)
        node.right = self._build(intervals, mid + 1, right)

        # Update covered_count to include children
        if node.left:
            node.covered_count += node.left.covered_count
        if node.right:
            node.covered_count += node.right.covered_count

        return node

    def delete_at_index(self, index):
        """Delete element at given index."""
        if self.root:
            self.root = self._delete_helper(self.root, index)

    def _delete_helper(self, node, index):
        """Recursively find and delete element."""
        if not node:
            return None

        left_count = node.left.covered_count if node.left else 0
        current_interval_size = node.end - node.start + 1

        if index < left_count:
            # Target is in left subtree
            node.left = self._delete_helper(node.left, index)
            node.covered_count -= 1
            return node

        elif index < left_count + current_interval_size:
            # Target is in current node's interval
            offset = index - left_count
            target_value = node.start + offset

            # Case 1: Deleting the only element in this interval
            if target_value == node.start and target_value == node.end:
                # Remove this node and merge its children
                return self._merge_children(node.left, node.right)

            # Case 2: Delete from start
            elif target_value == node.start:
                node.start += 1
                node.covered_count -= 1
                return node

            # Case 3: Delete from end
            elif target_value == node.end:
                node.end -= 1
                node.covered_count -= 1
                return node

            # Case 4: Delete from middle - split the interval
            else:
                # Create two new intervals: [start, target-1] and [target+1, end]
                left_interval = (node.start, target_value - 1)
                right_interval = (target_value + 1, node.end)

                # Replace current node with left interval
                node.start = left_interval[0]
                node.end = left_interval[1]
                node.covered_count = (node.end - node.start + 1)

                # Add children's counts
                if node.left:
                    node.covered_count += node.left.covered_count
                if node.right:
                    node.covered_count += node.right.covered_count

                # Insert the right interval into the tree
                # We need to add it to the right subtree
                new_right_node = IntervalNode(right_interval[0], right_interval[1],
                                               right_interval[1] - right_interval[0] + 1)
                new_right_node.right = node.right
                node.right = new_right_node

                # Update covered_count to include new right node
                node.covered_count += new_right_node.covered_count

                return node
        else:
            # Target is in right subtree
            right_index = index - left_count - current_interval_size
            node.right = self._delete_helper(node.right, right_index)
            node.covered_count -= 1
            return node

    def _merge_children(self, left, right):
        """Merge two subtrees when a node is deleted."""
        if not left:
            return right
        if not right:
            return left

        # Find the rightmost node in the left subtree
        # and attach the right subtree to it
        current = left
        while current.right:
            current = current.right
        current.right = right

        # Update covered_count for the path
        self._update_counts(left)

        return left

    def _update_counts(self, node):
        """Recalculate covered_count for a node and its subtree."""
        if not node:
            return 0

        left_count = self._update_counts(node.left)
        right_count = self._update_counts(node.right)
        current_count = node.end - node.start + 1

        node.covered_count = current_count + left_count + right_count
        return node.covered_count

    def to_intervals(self):
        """Convert tree back to interval list."""
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node, result):
        """In-order traversal to get sorted intervals."""
        if not node:
            return
        self._inorder(node.left, result)
        if node.start <= node.end:  # Valid interval
            result.append((node.start, node.end))
        self._inorder(node.right, result)

    def get_total_count(self):
        """Get total number of elements in all intervals."""
        return self.root.covered_count if self.root else 0

    def print_tree(self, node=None, level=0, prefix="Root: "):
        """Print tree structure for debugging."""
        if node is None:
            node = self.root
        if node is not None:
            print(" " * (level * 4) + prefix + f"({node.start}, {node.end}) count={node.covered_count}")
            if node.left:
                self.print_tree(node.left, level + 1, "L: ")
            if node.right:
                self.print_tree(node.right, level + 1, "R: ")


def test_interval_tree():
    print("=" * 60)
    print("Test 1: Delete from middle")
    print("=" * 60)
    intervals = [(4, 7), (10, 11), (13, 15)]
    tree = IntervalTree(intervals)
    print("Initial intervals:", tree.to_intervals())
    print("Initial tree structure:")
    tree.print_tree()
    print(f"\nTotal elements: {tree.get_total_count()}")

    # Delete index 2 (element 6)
    tree.delete_at_index(2)
    result = tree.to_intervals()
    print(f"\nAfter deleting index 2: {result}")
    print(f"Total elements: {tree.get_total_count()}")
    expected = [(4, 5), (7, 7), (10, 11), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 2: Delete from start")
    print("=" * 60)
    intervals = [(4, 7), (10, 11), (13, 15)]
    tree = IntervalTree(intervals)
    print("Initial intervals:", tree.to_intervals())

    # Delete index 0 (element 4)
    tree.delete_at_index(0)
    result = tree.to_intervals()
    print(f"After deleting index 0: {result}")
    expected = [(5, 7), (10, 11), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 3: Delete from end")
    print("=" * 60)
    intervals = [(4, 7), (10, 11), (13, 15)]
    tree = IntervalTree(intervals)
    print("Initial intervals:", tree.to_intervals())

    # Delete index 3 (element 7)
    tree.delete_at_index(3)
    result = tree.to_intervals()
    print(f"After deleting index 3: {result}")
    expected = [(4, 6), (10, 11), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 4: Delete single-element interval")
    print("=" * 60)
    intervals = [(4, 7), (10, 10), (13, 15)]
    tree = IntervalTree(intervals)
    print("Initial intervals:", tree.to_intervals())

    # Delete index 4 (element 10, which is a single-element interval)
    tree.delete_at_index(4)
    result = tree.to_intervals()
    print(f"After deleting index 4: {result}")
    expected = [(4, 7), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 5: Multiple deletions")
    print("=" * 60)
    intervals = [(1, 5), (10, 15)]
    tree = IntervalTree(intervals)
    print("Initial intervals:", tree.to_intervals())
    print(f"Total elements: {tree.get_total_count()}")

    # Elements: [1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15]
    # Indices:  [0, 1, 2, 3, 4,  5,  6,  7,  8,  9, 10]

    tree.delete_at_index(2)  # Delete 3
    print(f"After deleting index 2: {tree.to_intervals()}")

    tree.delete_at_index(0)  # Delete 1
    print(f"After deleting index 0: {tree.to_intervals()}")

    tree.delete_at_index(2)  # Delete 5 (was index 3, now index 2)
    print(f"After deleting index 2: {tree.to_intervals()}")
    print(f"Total elements: {tree.get_total_count()}")

    print("\n" + "=" * 60)
    print("All tests passed!")
    print("=" * 60)


if __name__ == "__main__":
    test_interval_tree()

Test 1: Delete from middle
Initial intervals: [(4, 7), (10, 11), (13, 15)]
Initial tree structure:
Root: (10, 11) count=9
    L: (4, 7) count=4
    R: (13, 15) count=3

Total elements: 9

After deleting index 2: [(4, 5), (7, 7), (10, 11), (13, 15)]
Total elements: 8

Test 2: Delete from start
Initial intervals: [(4, 7), (10, 11), (13, 15)]
After deleting index 0: [(5, 7), (10, 11), (13, 15)]

Test 3: Delete from end
Initial intervals: [(4, 7), (10, 11), (13, 15)]
After deleting index 3: [(4, 6), (10, 11), (13, 15)]

Test 4: Delete single-element interval
Initial intervals: [(4, 7), (10, 10), (13, 15)]
After deleting index 4: [(4, 7), (13, 15)]

Test 5: Multiple deletions
Initial intervals: [(1, 5), (10, 15)]
Total elements: 11
After deleting index 2: [(1, 2), (4, 5), (10, 15)]
After deleting index 0: [(2, 2), (4, 5), (10, 15)]
After deleting index 2: [(2, 2), (4, 4), (10, 15)]
Total elements: 8

All tests passed!


In [ ]:
## FOllow up 3 skipped list:
import random

class IntervalSkipNode:
    def __init__(self, start, end, level):
        self.start = start
        self.end = end
        self.covered_count = end - start + 1  # Elements in this interval
        self.forward = [None] * (level + 1)   # Forward pointers at each level
        self.span = [0] * (level + 1)          # Distance (in elements) to next node

class IntervalSkipList:
    def __init__(self, intervals=None, max_level=16, p=0.5):
        self.max_level = max_level
        self.p = p  # Probability for level generation
        self.header = IntervalSkipNode(-1, -1, max_level)
        self.level = 0  # Current max level in use

        if intervals:
            for start, end in intervals:
                self.insert(start, end)

    def _random_level(self):
        """Generate random level for new node."""
        level = 0
        while random.random() < self.p and level < self.max_level:
            level += 1
        return level

    def insert(self, start, end):
        """Insert a new interval into the skip list."""
        update = [None] * (self.max_level + 1)
        rank = [0] * (self.max_level + 1)  # Cumulative position at each level

        current = self.header
        cumulative = 0

        # Find position to insert
        for i in range(self.level, -1, -1):
            rank[i] = 0 if i == self.level else rank[i + 1]

            while current.forward[i]:
                rank[i] += current.span[i]
                current = current.forward[i]

            update[i] = current

        # Generate random level for new node
        new_level = self._random_level()

        # Update max level if necessary
        if new_level > self.level:
            for i in range(self.level + 1, new_level + 1):
                rank[i] = 0
                update[i] = self.header
                update[i].span[i] = self._get_total_count()
            self.level = new_level

        # Create new node
        new_node = IntervalSkipNode(start, end, new_level)

        # Insert node and update pointers
        for i in range(new_level + 1):
            new_node.forward[i] = update[i].forward[i]
            update[i].forward[i] = new_node

            # Update spans
            new_node.span[i] = update[i].span[i] - (rank[0] - rank[i])
            update[i].span[i] = (rank[0] - rank[i]) + new_node.covered_count

        # Update spans for levels not touched
        for i in range(new_level + 1, self.level + 1):
            update[i].span[i] += new_node.covered_count

    def find_at_index(self, index):
        """
        Find interval containing element at index.
        Returns: (node, start_index_of_node)
        """
        current = self.header
        cumulative = 0

        for i in range(self.level, -1, -1):
            while (current.forward[i] and
                   cumulative + current.span[i] <= index):
                cumulative += current.span[i]
                current = current.forward[i]

        # Move to actual node (skip header)
        if current == self.header and current.forward[0]:
            current = current.forward[0]
            cumulative = 0

        return current, cumulative

    def delete_at_index(self, index):
        """Delete element at given index."""
        update = [None] * (self.max_level + 1)
        current = self.header
        cumulative = 0

        # Find the node containing the index
        for i in range(self.level, -1, -1):
            while (current.forward[i] and
                   cumulative + current.span[i] <= index):
                cumulative += current.span[i]
                current = current.forward[i]
            update[i] = current

        # Get the actual node
        target_node = current.forward[0] if current == self.header else current
        if not target_node or target_node == self.header:
            return  # Index out of bounds

        # Calculate position within the interval
        node_start_index = cumulative if current != self.header else 0
        if current == self.header and current.forward[0]:
            node_start_index = 0
            target_node = current.forward[0]
            # Recalculate to find the right node
            current = self.header
            cumulative = 0
            for i in range(self.level, -1, -1):
                while (current.forward[i] and
                       cumulative + current.span[i] <= index):
                    cumulative += current.span[i]
                    current = current.forward[i]
                update[i] = current
            target_node = current.forward[0]
            node_start_index = cumulative

        if not target_node:
            return

        offset = index - node_start_index
        target_value = target_node.start + offset

        # Case 1: Single element interval - delete entire node
        if target_node.start == target_node.end:
            self._delete_node(target_node, update)

        # Case 2: Delete from start
        elif target_value == target_node.start:
            target_node.start += 1
            target_node.covered_count -= 1
            self._update_spans_after_shrink(update)

        # Case 3: Delete from end
        elif target_value == target_node.end:
            target_node.end -= 1
            target_node.covered_count -= 1
            self._update_spans_after_shrink(update)

        # Case 4: Delete from middle - split
        else:
            # Create new interval for right part
            right_start = target_value + 1
            right_end = target_node.end

            # Shrink current interval to left part
            target_node.end = target_value - 1
            old_count = target_node.covered_count
            target_node.covered_count = target_node.end - target_node.start + 1

            # Insert right interval
            self.insert(right_start, right_end)

            # Update spans (we removed one element)
            self._update_spans_after_shrink(update)

    def _delete_node(self, node, update):
        """Remove a node completely from the skip list."""
        for i in range(self.level + 1):
            if update[i].forward[i] == node:
                update[i].span[i] += node.span[i] - node.covered_count
                update[i].forward[i] = node.forward[i]
            else:
                update[i].span[i] -= node.covered_count

        # Update level if necessary
        while self.level > 0 and self.header.forward[self.level] is None:
            self.level -= 1

    def _update_spans_after_shrink(self, update):
        """Update spans after reducing an interval by 1."""
        for i in range(self.level + 1):
            if update[i].forward[i]:
                update[i].span[i] -= 1

    def _get_total_count(self):
        """Get total number of elements in all intervals."""
        total = 0
        current = self.header.forward[0]
        while current:
            total += current.covered_count
            current = current.forward[0]
        return total

    def to_intervals(self):
        """Convert skip list to interval list."""
        result = []
        current = self.header.forward[0]
        while current:
            result.append((current.start, current.end))
            current = current.forward[0]
        return result

    def print_structure(self):
        """Print skip list structure for debugging."""
        print(f"Skip List (level={self.level}):")
        for level in range(self.level, -1, -1):
            print(f"Level {level}: ", end="")
            current = self.header
            while current.forward[level]:
                current = current.forward[level]
                print(f"({current.start},{current.end})[span={current.span[level]}] -> ", end="")
            print("None")
        print()


def test_skip_list():
    print("=" * 60)
    print("Test 1: Basic insertion and display")
    print("=" * 60)
    random.seed(42)  # For reproducible results
    intervals = [(4, 7), (10, 11), (13, 15)]
    skip_list = IntervalSkipList(intervals)
    print("Intervals:", skip_list.to_intervals())
    skip_list.print_structure()

    print("=" * 60)
    print("Test 2: Delete from middle")
    print("=" * 60)
    random.seed(42)
    intervals = [(4, 7), (10, 11), (13, 15)]
    skip_list = IntervalSkipList(intervals)
    print("Initial:", skip_list.to_intervals())

    # Elements: [4, 5, 6, 7, 10, 11, 13, 14, 15]
    # Indices:  [0, 1, 2, 3,  4,  5,  6,  7,  8]
    # Delete index 2 (element 6)
    skip_list.delete_at_index(2)
    result = skip_list.to_intervals()
    print("After deleting index 2:", result)
    # Expected: [(4, 5), (7, 7), (10, 11), (13, 15)]

    print("\n" + "=" * 60)
    print("Test 3: Delete from start")
    print("=" * 60)
    random.seed(42)
    intervals = [(4, 7), (10, 11), (13, 15)]
    skip_list = IntervalSkipList(intervals)
    print("Initial:", skip_list.to_intervals())

    # Delete index 0 (element 4)
    skip_list.delete_at_index(0)
    result = skip_list.to_intervals()
    print("After deleting index 0:", result)
    expected = [(5, 7), (10, 11), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 4: Delete from end")
    print("=" * 60)
    random.seed(42)
    intervals = [(4, 7), (10, 11), (13, 15)]
    skip_list = IntervalSkipList(intervals)
    print("Initial:", skip_list.to_intervals())

    # Delete index 3 (element 7)
    skip_list.delete_at_index(3)
    result = skip_list.to_intervals()
    print("After deleting index 3:", result)
    expected = [(4, 6), (10, 11), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 5: Delete single-element interval")
    print("=" * 60)
    random.seed(42)
    intervals = [(4, 7), (10, 10), (13, 15)]
    skip_list = IntervalSkipList(intervals)
    print("Initial:", skip_list.to_intervals())

    # Elements: [4, 5, 6, 7, 10, 13, 14, 15]
    # Indices:  [0, 1, 2, 3,  4,  5,  6,  7]
    # Delete index 4 (element 10)
    skip_list.delete_at_index(4)
    result = skip_list.to_intervals()
    print("After deleting index 4:", result)
    expected = [(4, 7), (13, 15)]
    assert result == expected, f"Expected {expected}, got {result}"

    print("\n" + "=" * 60)
    print("Test 6: Multiple consecutive deletions")
    print("=" * 60)
    random.seed(42)
    intervals = [(1, 5), (10, 15)]
    skip_list = IntervalSkipList(intervals)
    print("Initial:", skip_list.to_intervals())

    # Elements: [1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15]
    # Indices:  [0, 1, 2, 3, 4,  5,  6,  7,  8,  9, 10]

    skip_list.delete_at_index(2)  # Delete 3
    print("After deleting index 2:", skip_list.to_intervals())

    skip_list.delete_at_index(0)  # Delete 1
    print("After deleting index 0:", skip_list.to_intervals())

    skip_list.delete_at_index(2)  # Delete 5 (was index 3, now 2)
    print("After deleting index 2:", skip_list.to_intervals())

    print("\n" + "=" * 60)
    print("All tests passed!")
    print("=" * 60)


test_skip_list()


Test 1: Basic insertion and display
Intervals: [(4, 7), (10, 11), (13, 15)]
Skip List (level=3):
Level 3: (10,11)[span=3] -> None
Level 2: (10,11)[span=3] -> None
Level 1: (10,11)[span=3] -> None
Level 0: (4,7)[span=2] -> (10,11)[span=3] -> (13,15)[span=0] -> None

Test 2: Delete from middle
Initial: [(4, 7), (10, 11), (13, 15)]
After deleting index 2: [(4, 5), (10, 11), (13, 15), (7, 7)]

Test 3: Delete from start
Initial: [(4, 7), (10, 11), (13, 15)]
After deleting index 0: [(5, 7), (10, 11), (13, 15)]

Test 4: Delete from end
Initial: [(4, 7), (10, 11), (13, 15)]
After deleting index 3: [(4, 6), (10, 11), (13, 15)]

Test 5: Delete single-element interval
Initial: [(4, 7), (10, 10), (13, 15)]
After deleting index 4: [(5, 7), (10, 10), (13, 15)]


AssertionError: Expected [(4, 7), (13, 15)], got [(5, 7), (10, 10), (13, 15)]

In [ ]:

def insert_into_intervals(intervals, target_index, value):
    """
    Insert a value at the given index in the intervals.
    The value is inserted BEFORE the element currently at target_index.

    Args:
        intervals: List of tuples (start, end) representing ranges
        target_index: 0-based index where to insert (inserts before this position)
        value: The integer value to insert

    Returns:
        New list of intervals with the value inserted
    """
    if not intervals:
        return [(value, value)]

    # Special case: insert at the very end (after all elements)
    total_count = sum(end - start + 1 for start, end in intervals)
    if target_index >= total_count:
        # Check if we can extend the last interval
        last_start, last_end = intervals[-1]
        if value == last_end + 1:
            return intervals[:-1] + [(last_start, last_end + 1)]
        else:
            return intervals + [(value, value)]

    new_intervals = []
    running_count = 0

    for i, interval in enumerate(intervals):
        start, end = interval[0], interval[1]
        count = end - start + 1
        prev_count = running_count
        running_count += count

        # If target_index is beyond this interval, keep it unchanged
        if target_index >= running_count:
            new_intervals.append((start, end))
        # target_index is within or at the start of this interval
        else:
            # Position within current interval (0-based)
            pos_in_interval = target_index - prev_count
            insert_before = start + pos_in_interval

            # Case 1: Insert at the very beginning of this interval
            if pos_in_interval == 0:
                # Try to extend previous interval
                if new_intervals and new_intervals[-1][1] == value - 1:
                    prev_start, prev_end = new_intervals[-1]
                    new_intervals[-1] = (prev_start, value)
                    # Check if we can merge with current interval
                    if value + 1 == start:
                        new_intervals[-1] = (prev_start, end)
                    else:
                        new_intervals.append((start, end))
                # Check if we can extend current interval from the left
                elif value == start - 1:
                    new_intervals.append((value, end))
                # Otherwise, insert as standalone
                else:
                    new_intervals.append((value, value))
                    new_intervals.append((start, end))
            # Case 2: Insert in the middle of this interval
            else:
                # Split the interval and insert the value
                # Left part: [start, insert_before - 1]
                new_intervals.append((start, insert_before - 1))

                # Check if value can extend the left part
                if value == insert_before - 1:
                    # Merge with left part
                    new_intervals[-1] = (start, value)
                    # Check if we can merge with right part
                    if value + 1 == insert_before:
                        new_intervals[-1] = (start, end)
                    else:
                        new_intervals.append((insert_before, end))
                # Check if value can extend the right part
                elif value == insert_before:
                    # This shouldn't happen in normal cases, but handle it
                    new_intervals.append((value, end))
                # Value creates a gap
                else:
                    new_intervals.append((value, value))
                    new_intervals.append((insert_before, end))

            # Add all remaining intervals unchanged
            new_intervals.extend(intervals[i + 1:])
            break

    return new_intervals


def merge_intervals(intervals):
    """
    Helper function to merge adjacent or overlapping intervals.

    Args:
        intervals: List of tuples (start, end)

    Returns:
        Merged list of intervals
    """
    if not intervals:
        return []

    # Sort intervals by start position
    sorted_intervals = sorted(intervals)
    merged = [sorted_intervals[0]]

    for current in sorted_intervals[1:]:
        last_start, last_end = merged[-1]
        curr_start, curr_end = current

        # Check if intervals overlap or are adjacent
        if curr_start <= last_end + 1:
            # Merge them
            merged[-1] = (last_start, max(last_end, curr_end))
        else:
            # No overlap, add as new interval
            merged.append(current)

    return merged


def flatten_intervals(intervals):
    """
    Helper function to convert intervals to a flat list for visualization.

    Args:
        intervals: List of tuples (start, end)

    Returns:
        List of all integers in the intervals
    """
    result = []
    for start, end in intervals:
        result.extend(range(start, end + 1))
    return result


In [ ]:
def minimize_intervals_with_k_additions(intervals, k):
    """
    Add up to K elements to minimize number of intervals.

    Returns:
        - Minimum number of intervals achievable
        - List of values to add
    """
    if len(intervals) <= 1:
        return len(intervals), []

    # Calculate gaps between consecutive intervals
    gaps = []
    for i in range(len(intervals) - 1):
        end_current = intervals[i][1]
        start_next = intervals[i + 1][0]
        gap_size = start_next - end_current - 1

        if gap_size > 0:
            gaps.append({
                'size': gap_size,
                'start': end_current + 1,
                'end': start_next - 1,
                'index': i
            })

    # Sort by gap size (prioritize small gaps)
    gaps.sort(key=lambda g: g['size'])

    # Greedily fill gaps
    elements_to_add = []
    merged_count = 0

    for gap in gaps:
        if k >= gap['size']:
            # Fill entire gap
            elements_to_add.extend(range(gap['start'], gap['end'] + 1))
            k -= gap['size']
            merged_count += 1
        else:
            # Partial fill (may not merge intervals)
            break

    min_intervals = len(intervals) - merged_count
    return min_intervals, elements_to_add

# Example
intervals = [(1, 3), (5, 7), (9, 11), (15, 20)]
min_count, to_add = minimize_intervals_with_k_additions(intervals, 2)
print(f"Minimum intervals: {min_count}")  # 3
print(f"Add elements: {to_add}")  # [4, 8]

Minimum intervals: 2
Add elements: [4, 8]
